In [0]:
USE CATALOG olist_retail;
USE SCHEMA gold_dataset;
-- ==================================================================
-- Q1. Monthly Revenue Trend
-- Q: How does revenue fluctuate month-to-month?
-- ==================================================================
SELECT
  date_format(order_date, 'yyyy-MM') AS month_year,
  COUNT(DISTINCT order_id) AS total_orders,
  ROUND(SUM(payment_value),2) AS monthly_revenue
FROM fact_sales
GROUP BY month_year
ORDER BY month_year;

-- ==================================================================
-- Q2. Top Product Categories by Revenue
-- Q: Which product categories generate the most revenue?
-- ==================================================================
SELECT
  p.product_category_name,
  ROUND(SUM(fs.payment_value),2) AS category_revenue
FROM fact_sales fs
JOIN dim_product p ON fs.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY category_revenue DESC
LIMIT 10;

-- ==================================================================
-- Q3. Revenue by State
-- Q: Which customer states contribute most to overall revenue?
-- ==================================================================
SELECT
  c.customer_state,
  ROUND(SUM(fs.payment_value),2) AS state_revenue
FROM fact_sales fs
JOIN dim_customer c ON fs.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY state_revenue DESC;

-- ==================================================================
-- Q4. Seller Performance by Revenue
-- Q: Who are the highest earning sellers?
-- ==================================================================
SELECT
  s.seller_id,
  s.seller_city,
  s.seller_state,
  ROUND(SUM(fs.payment_value),2) AS seller_revenue
FROM fact_sales fs
JOIN dim_seller s ON fs.seller_id = s.seller_id
GROUP BY s.seller_id, s.seller_city, s.seller_state
ORDER BY seller_revenue DESC
LIMIT 10;

-- ==================================================================
-- Q5. Payment Type Influence on Revenue
-- Q: How does payment method impact revenue distribution?
-- ==================================================================
SELECT
  payment_type,
  ROUND(SUM(payment_value),2) AS payment_revenue,
  COUNT(DISTINCT order_id) AS num_orders
FROM fact_sales
GROUP BY payment_type
ORDER BY payment_revenue DESC;

-- ==================================================================
-- CUSTOMER ANALYSIS
-- ==================================================================

-- Q6. Customer Lifetime Value Analysis
-- Q: What is the average order value and repeat purchase rate?
SELECT
  c.customer_state,
  COUNT(DISTINCT fs.customer_id) AS total_customers,
  COUNT(DISTINCT fs.order_id) AS total_orders,
  ROUND(AVG(fs.payment_value),2) AS avg_order_value,
  ROUND(COUNT(DISTINCT fs.order_id) * 1.0 / COUNT(DISTINCT fs.customer_id),2) AS orders_per_customer
FROM fact_sales fs
JOIN dim_customer c ON fs.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_customers DESC
LIMIT 10;

-- Q7. Top Customers by Revenue
-- Q: Who are our most valuable customers?
SELECT
  fs.customer_id,
  c.customer_city,
  c.customer_state,
  COUNT(DISTINCT fs.order_id) AS num_orders,
  ROUND(SUM(fs.payment_value),2) AS customer_revenue,
  ROUND(AVG(fs.review_score),1) AS avg_review_score
FROM fact_sales fs
JOIN dim_customer c ON fs.customer_id = c.customer_id
GROUP BY fs.customer_id, c.customer_city, c.customer_state
ORDER BY customer_revenue DESC
LIMIT 10;

-- Q8. Customer Geographic Distribution
-- Q: What is the revenue concentration by customer location?
SELECT
  c.customer_state,
  COUNT(DISTINCT c.customer_city) AS num_cities,
  COUNT(DISTINCT fs.customer_id) AS num_customers,
  ROUND(SUM(fs.payment_value),2) AS state_revenue,
  ROUND(AVG(fs.payment_value),2) AS avg_order_value
FROM fact_sales fs
JOIN dim_customer c ON fs.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY state_revenue DESC;

-- ==================================================================
-- SELLER & PRODUCTS ANALYSIS
-- ==================================================================

-- Q9. Seller Efficiency Metrics
-- Q: Which sellers have the best order volume and revenue efficiency?
SELECT
  s.seller_id,
  s.seller_state,
  COUNT(DISTINCT fs.order_id) AS total_orders,
  ROUND(SUM(fs.payment_value),2) AS total_revenue,
  ROUND(AVG(fs.payment_value),2) AS avg_order_value,
  ROUND(AVG(fs.review_score),1) AS avg_review_score
FROM fact_sales fs
JOIN dim_seller s ON fs.seller_id = s.seller_id
GROUP BY s.seller_id, s.seller_state
HAVING COUNT(DISTINCT fs.order_id) >= 10
ORDER BY total_revenue DESC
LIMIT 10;

-- Q10. Product Category Performance Over Time
-- Q: How do product categories perform across different months?
SELECT
  date_format(fs.order_date, 'yyyy-MM') AS month_year,
  p.product_category_name,
  COUNT(DISTINCT fs.order_id) AS orders,
  ROUND(SUM(fs.payment_value),2) AS revenue
FROM fact_sales fs
JOIN dim_product p ON fs.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY month_year, p.product_category_name
ORDER BY month_year, revenue DESC;

-- Q11. Product Pricing Analysis
-- Q: What is the relationship between price, freight, and total payment?
SELECT
  p.product_category_name,
  COUNT(DISTINCT fs.order_id) AS num_orders,
  ROUND(AVG(fs.price),2) AS avg_product_price,
  ROUND(AVG(fs.freight_value),2) AS avg_freight,
  ROUND(AVG(fs.payment_value),2) AS avg_total_payment,
  ROUND((AVG(fs.freight_value) / NULLIF(AVG(fs.price),0)) * 100,2) AS freight_to_price_pct
FROM fact_sales fs
JOIN dim_product p ON fs.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY p.product_category_name
ORDER BY num_orders DESC
LIMIT 15;

-- Q12. Seller Performance by State
-- Q: Which states have the most successful sellers?
SELECT
  s.seller_state,
  COUNT(DISTINCT s.seller_id) AS num_sellers,
  COUNT(DISTINCT fs.order_id) AS total_orders,
  ROUND(SUM(fs.payment_value),2) AS total_revenue,
  ROUND(AVG(fs.payment_value),2) AS avg_order_value
FROM fact_sales fs
JOIN dim_seller s ON fs.seller_id = s.seller_id
GROUP BY s.seller_state
ORDER BY total_revenue DESC;

-- ==================================================================
-- LOGISTICS ANALYSIS
-- ==================================================================

-- Q13. Delivery Performance by State
-- Q: How does delivery time vary across different states?
SELECT
  c.customer_state,
  COUNT(DISTINCT fs.order_id) AS total_orders,
  ROUND(AVG(fs.delivery_days),1) AS avg_delivery_days,
  MIN(fs.delivery_days) AS min_delivery_days,
  MAX(fs.delivery_days) AS max_delivery_days,
  ROUND(AVG(fs.review_score),1) AS avg_review_score
FROM fact_sales fs
JOIN dim_customer c ON fs.customer_id = c.customer_id
WHERE fs.delivery_days IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC;

-- Q14. Freight Cost Analysis
-- Q: How does freight value impact overall revenue by product category?
SELECT
  p.product_category_name,
  ROUND(SUM(fs.freight_value),2) AS total_freight_cost,
  ROUND(SUM(fs.payment_value),2) AS total_revenue,
  ROUND((SUM(fs.freight_value) / NULLIF(SUM(fs.payment_value),0)) * 100,2) AS freight_cost_percentage,
  ROUND(AVG(fs.freight_value),2) AS avg_freight_per_order
FROM fact_sales fs
JOIN dim_product p ON fs.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY p.product_category_name
ORDER BY total_freight_cost DESC
LIMIT 15;

-- Q15. Delivery Time Impact on Customer Satisfaction
-- Q: How does delivery speed correlate with review scores?
SELECT
  CASE
    WHEN delivery_days <= 7 THEN '1-7 days'
    WHEN delivery_days <= 14 THEN '8-14 days'
    WHEN delivery_days <= 21 THEN '15-21 days'
    ELSE '22+ days'
  END AS delivery_bucket,
  COUNT(DISTINCT order_id) AS total_orders,
  ROUND(AVG(review_score),2) AS avg_review_score,
  ROUND(AVG(payment_value),2) AS avg_order_value
FROM fact_sales
WHERE delivery_days IS NOT NULL
GROUP BY delivery_bucket
ORDER BY avg_review_score DESC;